# 🚀 Notebook 4: Performance Tuning, Skew Mitigation & Production ETL Patterns
### *From Partition Sizing and Salting to Adaptive Query Execution (AQE) and Production Pipelines*

> **Companion Video Reference:** [YouTube: PySpark Tutorial | Full Course](https://www.youtube.com/watch?v=94w6hPk7nkM) by Ansh Lamba & [Advancing Analytics Performance Optimization](https://www.youtube.com/@AdvancingAnalytics)

---

## 🎯 What Will You Learn in This Notebook?

In distributed computing, writing code that produces correct results is only step 1. Writing code that **scales to 100 million or 1 billion rows without crashing or freezing at 99%** is what separates junior developers from senior data engineers.

In this capstone notebook, we master:
1. **Partitioning Mechanics:** Sizing partitions and comparing `repartition()` vs. `coalesce()`.
2. **The "Straggler" Problem & Data Skew:** Why 1 task freezes for 2 hours while all others finish in 2 seconds.
3. **The Salting Technique:** How to defeat data skew mathematically using random salt keys.
4. **Caching & Persistence:** `cache()` vs. `persist(StorageLevel)` and avoiding memory leaks.
5. **Adaptive Query Execution (AQE):** How Spark 3.x and 4.x re-plans queries dynamically at runtime.
6. **The Python UDF Bottleneck:** Why standard Python UDFs ruin Catalyst optimization.
7. **End-to-End Production ETL Scenario:** Building an industry-standard, partitioned Parquet pipeline.

---


## 📦 1. Partitioning Mechanics: `repartition()` vs. `coalesce()`

Partitions govern **parallelism** in Apache Spark:
- **1 Partition = 1 Task = 1 CPU Core at any instant.**
- **Too Few Partitions (e.g. 1-2):** Most cluster CPU cores sit idle. High risk of `OutOfMemoryError` (OOM) as partitions exceed RAM.
- **Too Many Partitions (e.g. 50,000 for a 1 GB file):** Driver overhead explodes scheduling thousands of tiny tasks ("small files problem").

### The Crucial Difference:

```
REPARTITION(N) (Wide Shuffle):
• Performs a FULL network shuffle across the cluster.
• Can INCREASE or DECREASE partition count.
• Distributes rows uniformly across all output partitions (fixes uneven sizing).

=====================================================================

COALESCE(N) (Narrow Merge):
• Merges adjacent local partitions on the same executor WITHOUT a full shuffle!
• Can ONLY DECREASE partition count.
• Fast, but can create uneven partition sizes if upstream data was skewed.
```

> 💡 **Golden Rule:** When saving files to disk after a filter/join, use `df.coalesce(10).write...` to avoid writing thousands of tiny files without triggering an expensive full shuffle!


## ⚖️ 2. Data Skew & The Straggler Problem

### What is Data Skew?
In distributed databases, when data is partitioned across a cluster using a key (e.g. `country` or `seller_id`), real-world data is almost never evenly distributed:
- 90% of your transactions might belong to `country = "US"` or `seller_id = "AMAZON"`.
- 10% belongs to 100 other small countries.

```
THE STRAGGLER PROBLEM (Unbalanced Shuffle):
Executor 1 (Tasks for UK, CA, AU): [|||||] Done in 3 seconds!
Executor 2 (Tasks for DE, FR, JP): [|||||] Done in 3 seconds!
Executor 3 (Task for US - 90% data): [|||||||||||||||||||||||||||||||||||||||||||||...] Stuck at 99% for 4 hours!
```

### The Solution: Salting
**Salting** is the industry standard technique to defeat data skew:
1. Append a random integer (the **"salt"**, e.g., $0$ to $k-1$) to the skewed key: `concat(col("key"), lit("_"), floor(rand() * 4))`.
2. Group or join on `skewed_key_salted`. The single massive partition is now sliced into $k$ balanced sub-partitions running across $k$ executors in parallel!
3. Re-aggregate without the salt in a second quick pass.


## 🧠 3. Adaptive Query Execution (AQE) in Modern Spark

Introduced in Spark 3.0+ and enhanced in Spark 3.5/4.x, **AQE** is one of Spark's greatest architectural leaps:
- Traditional optimizers formulate a plan *before* executing. But until data is read, Spark doesn't know exact row counts or partition sizes.
- **AQE pauses execution at shuffle stage boundaries**, inspects real runtime metrics (actual stage outputs), and **dynamically re-optimizes the remaining plan**:
  1. **Dynamic Partition Coalescing:** Shrinks the default 200 shuffle partitions down to a smaller, optimal count automatically.
  2. **Dynamic Join Switching:** If an intermediate shuffle output is smaller than 10 MB, AQE automatically converts a slow Sort-Merge Join to a fast **Broadcast Hash Join** on the fly!
  3. **Dynamic Skew Join Handling:** Automatically detects skewed partitions and splits them into smaller sub-tasks.


## 🚀 Hands-On Lab: Session Setup with AQE Enabled


In [1]:
import os
import sys
from pathlib import Path

# Configure environment
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
from pyspark.sql.functions import (
    col,
    lit,
    rand,
    floor,
    concat,
    split,
    sum as spark_sum,
    count,
    avg,
    round as spark_round,
    row_number
)

# Initialize SparkSession with AQE enabled and custom shuffle partitions
spark = SparkSession.builder \
    .appName("PySpark_Performance_Tuning_and_Patterns") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession initialized with Adaptive Query Execution (AQE) enabled!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/22 11:41:11 WARN Utils: Your hostname, Prafull-Mac.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.161 instead (on interface en0)
26/09/22 11:41:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/Users/prafullsaxena/Desktop/Learning/Machine Learning/milvus/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/22 11:41:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession initialized with Adaptive Query Execution (AQE) enabled!


## 🔬 4. Comparing `repartition()` vs. `coalesce()`

Let's inspect how Spark alters partition layouts with `repartition()` vs `coalesce()`:


In [2]:
# Create a small DataFrame with 8 partitions
base_data = [(i, f"user_{i}", i % 5) for i in range(1, 1001)]
initial_df = spark.createDataFrame(base_data, ["id", "username", "group_id"]).repartition(8)

print(f"Initial Partition Count: {initial_df.rdd.getNumPartitions()}")

# 1. Repartition (Forces full network shuffle)
repartitioned_df = initial_df.repartition(4)
print(f"After repartition(4):   {repartitioned_df.rdd.getNumPartitions()}")

# 2. Coalesce (Merges local partitions without full shuffle)
coalesced_df = initial_df.coalesce(2)
print(f"After coalesce(2):      {coalesced_df.rdd.getNumPartitions()}")

print("\n🔍 Execution plan for coalesce(2) - Notice ZERO Shuffle Exchange:")
coalesced_df.explain()


Initial Partition Count: 8
After repartition(4):   4
After coalesce(2):      2

🔍 Execution plan for coalesce(2) - Notice ZERO Shuffle Exchange:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 2
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=52]
            +- *(1) Scan ExistingRDD[id#0L,username#1,group_id#2L]
+- == Initial Plan ==
   Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=45]
      +- Scan ExistingRDD[id#0L,username#1,group_id#2L]




## 🧂 5. Mitigating Data Skew Using Salting

Let's simulate a heavily skewed dataset where 85% of records belong to `"US"` and demonstrate how salting balances the workload:


In [3]:
# Simulating a heavily skewed dataset: 850 rows US, 50 rows each for others
skewed_records = [("US", 10.0)] * 850 + [("CA", 15.0)] * 50 + [("UK", 20.0)] * 50 + [("DE", 25.0)] * 50
skewed_df = spark.createDataFrame(skewed_records, ["country", "amount"])

print("📊 Distribution before salting:")
skewed_df.groupBy("country").count().show()

# --- THE SALTING TECHNIQUE ---
# Step 1: Add random salt (0 to 3) to skewed keys to split the partition
salted_df = skewed_df.withColumn("salt", floor(rand() * 4)) \
                     .withColumn("salted_country", concat(col("country"), lit("_salt_"), col("salt")))

# Step 2: First-stage aggregation on salted key (work is distributed across 4x tasks!)
stage1_agg = salted_df.groupBy("salted_country") \
                      .agg(
                          spark_sum("amount").alias("partial_sum"),
                          count("amount").alias("partial_count")
                      )

# Step 3: Strip the salt and do final second-stage aggregation
final_agg = stage1_agg.withColumn("country", split(col("salted_country"), "_salt_").getItem(0)) \
                      .groupBy("country") \
                      .agg(
                          spark_sum("partial_sum").alias("total_revenue"),
                          spark_sum("partial_count").alias("total_transactions")
                      ).orderBy("total_revenue", ascending=False)

print("✅ Balanced Aggregation Result via Salting:")
final_agg.show()


📊 Distribution before salting:


+-------+-----+
|country|count|
+-------+-----+
|     US|  850|
|     CA|   50|
|     DE|   50|
|     UK|   50|
+-------+-----+

✅ Balanced Aggregation Result via Salting:


+-------+-------------+------------------+
|country|total_revenue|total_transactions|
+-------+-------------+------------------+
|     US|       8500.0|               850|
|     DE|       1250.0|                50|
|     UK|       1000.0|                50|
|     CA|        750.0|                50|
+-------+-------------+------------------+



## 💾 6. Caching & Persistence Strategies

When should you call `.cache()`?
- **When to Cache:** If an intermediate DataFrame is evaluated **multiple times** by different downstream actions (e.g., training an ML model with 10 iterations, or saving a cleaned table to 3 different formats).
- **When NOT to Cache:** If the DataFrame is only used once. Caching it wastes RAM and incurs serialization overhead.
- **Always Clean Up:** Call `df.unpersist()` once the downstream actions finish to prevent memory leaks!


In [4]:
# Create expensive intermediate calculation
calc_df = skewed_df.groupBy("country").agg(spark_sum("amount").alias("rev"))

# Persist in RAM and Disk
calc_df.persist(StorageLevel.MEMORY_AND_DISK)

# Action 1: Count
print(f"Action 1 (Count): {calc_df.count()}")

# Action 2: Show (Reuses cached memory, does NOT recompute from scratch!)
print("Action 2 (Show - Served from Cache):")
calc_df.show()

# Free memory!
calc_df.unpersist()
print("✅ Unpersisted DataFrame from RAM/Disk!")


Action 1 (Count): 4
Action 2 (Show - Served from Cache):
+-------+------+
|country|   rev|
+-------+------+
|     CA| 750.0|
|     DE|1250.0|
|     US|8500.0|
|     UK|1000.0|
+-------+------+

✅ Unpersisted DataFrame from RAM/Disk!


## 🏭 7. Capstone: Production ETL Pipeline

Let's put everything we've learned across all 4 notebooks into an **end-to-end production ETL pipeline**:
1. **Extract:** Ingest raw multi-channel e-commerce events.
2. **Transform:** Validate schemas, handle nulls, enrich via Broadcast Join, and compute Customer Lifetime Value (CLV) with Window functions.
3. **Load:** Write partitioned Parquet files partitioned by `country` for optimized downstream data lake querying!


In [5]:
# 1. Raw Transactions Stream
raw_orders = [
    ("ORD_101", "C_1", "2024-01-15", 150.0, "US"),
    ("ORD_102", "C_2", "2024-01-16", 45.0, "UK"),
    ("ORD_103", "C_1", "2024-01-17", 320.0, "US"),
    ("ORD_104", "C_3", "2024-01-18", 1200.0, "CA"),
    ("ORD_105", "C_2", "2024-01-19", 90.0, "UK"),
    ("ORD_106", "C_1", "2024-01-20", 210.0, "US"),
    ("ORD_107", "C_4", None, 50.0, "US"),          # Bad record: null date
    ("ORD_108", "C_5", "2024-01-22", None, "CA"),  # Bad record: null amount
]

orders_df = spark.createDataFrame(raw_orders, ["order_id", "cust_id", "order_date", "amount", "country"])

# 2. Customer Dimension
customers = [
    ("C_1", "Enterprise Tier", 0.15),
    ("C_2", "Growth Tier", 0.05),
    ("C_3", "Enterprise Tier", 0.15),
    ("C_4", "Standard Tier", 0.0),
    ("C_5", "Standard Tier", 0.0),
]
cust_df = spark.createDataFrame(customers, ["cust_id", "tier", "discount_rate"])

# --- ETL TRANSFORMATIONS ---
# Step A: Clean bad records
valid_orders = orders_df.na.drop(subset=["order_date", "amount"])

# Step B: Broadcast Join with Customer Dimension (Zero network shuffle!)
from pyspark.sql.functions import broadcast
enriched_df = valid_orders.join(broadcast(cust_df), "cust_id", "left")

# Step C: Compute Net Amount after Discount
enriched_df = enriched_df.withColumn(
    "net_amount",
    spark_round(col("amount") * (lit(1.0) - col("discount_rate")), 2)
)

# Step D: Compute Customer Lifetime Spend & Order Sequence using Window Function
cust_window = Window.partitionBy("cust_id").orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

order_seq_window = Window.partitionBy("cust_id").orderBy("order_date")

final_curated_df = enriched_df.withColumn(
    "order_sequence_num",
    row_number().over(order_seq_window)
).withColumn(
    "customer_cumulative_spend",
    spark_round(spark_sum("net_amount").over(cust_window), 2)
)

print("✅ Final Curated ETL Dataset:")
final_curated_df.show(truncate=False)

# Step E: Load to Partitioned Parquet Lakehouse
OUTPUT_LAKE_PATH = "./data/lakehouse/curated_orders"
final_curated_df.write \
    .mode("overwrite") \
    .partitionBy("country") \
    .parquet(OUTPUT_LAKE_PATH)

print(f"🎉 Production Lakehouse table successfully written to: {OUTPUT_LAKE_PATH}")


✅ Final Curated ETL Dataset:


+-------+--------+----------+------+-------+---------------+-------------+----------+------------------+-------------------------+
|cust_id|order_id|order_date|amount|country|tier           |discount_rate|net_amount|order_sequence_num|customer_cumulative_spend|
+-------+--------+----------+------+-------+---------------+-------------+----------+------------------+-------------------------+
|C_1    |ORD_101 |2024-01-15|150.0 |US     |Enterprise Tier|0.15         |127.5     |1                 |127.5                    |
|C_1    |ORD_103 |2024-01-17|320.0 |US     |Enterprise Tier|0.15         |272.0     |2                 |399.5                    |
|C_1    |ORD_106 |2024-01-20|210.0 |US     |Enterprise Tier|0.15         |178.5     |3                 |578.0                    |
|C_2    |ORD_102 |2024-01-16|45.0  |UK     |Growth Tier    |0.05         |42.75     |1                 |42.75                    |
|C_2    |ORD_105 |2024-01-19|90.0  |UK     |Growth Tier    |0.05         |85.5     

🎉 Production Lakehouse table successfully written to: ./data/lakehouse/curated_orders


### Clean Session Shutdown


In [6]:
# Cleanly shutdown SparkSession
spark.stop()
print("✅ SparkSession cleanly terminated. Resources freed!")


✅ SparkSession cleanly terminated. Resources freed!


## 📖 Complete Production Spark Performance Checklist

Before deploying any PySpark job to production, verify:

1. **Did you avoid `inferSchema=True`?** $\rightarrow$ Always pass explicit `StructType`.
2. **Are small dimension tables broadcasted?** $\rightarrow$ Use `join(broadcast(dim_df), ...)`.
3. **Is Data Skew mitigated?** $\rightarrow$ Check Spark UI for straggler tasks; apply **Salting** if needed.
4. **Is AQE enabled?** $\rightarrow$ Ensure `spark.sql.adaptive.enabled = true`.
5. **Did you avoid standard Python UDFs?** $\rightarrow$ Use built-in `pyspark.sql.functions` or Vectorized Pandas UDFs.
6. **Are you writing thousands of tiny files?** $\rightarrow$ Call `.coalesce(N)` before `.write.parquet(...)`.
7. **Did you unpersist cached DataFrames?** $\rightarrow$ Always call `df.unpersist()` after downstream actions.

---
### 🎓 Congratulations! You now understand Apache Spark from basic syntax to cluster mechanics and production performance tuning!
